# LLM-KGKAN: Full Paper ReproductionReproduces all **16 tables** and **5 figures**.

In [ ]:
import subprocess, syssubprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements2.txt"])try:    import spacy; spacy.load("en_core_web_sm")except:    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])print("✅ Dependencies installed")

In [ ]:
import os, sys, json, time, gc, warnings, randomimport numpy as npimport pandas as pdimport torchwarnings.filterwarnings("ignore")sys.path.insert(0, os.getcwd())from scripts.config import *from scripts.data_utils import *from scripts.evaluate import *from scripts.train_all import *from scripts.llm_inference import *from scripts.visualize import *set_seed(SEED)device = torch.device(DEVICE)print(f"Device: {device}")print(f"GPU: {GPU.name} ({GPU.vram_gb}GB)")print(f"API budget: ${API_BUDGET.max_budget_usd}")

## Phase 0: Knowledge Graph Setup

In [ ]:
# Load ConceptNetfrom kg_utils import ConceptNetCN_PATH = "conceptnet-assertions-5.7.0.csv"if not os.path.exists(CN_PATH):    CN_GZ = CN_PATH + ".gz"    if os.path.exists(CN_GZ):        import gzip, shutil        print("Extracting ConceptNet...")        with gzip.open(CN_GZ, 'rb') as f_in:            with open(CN_PATH, 'wb') as f_out:                shutil.copyfileobj(f_in, f_out)    else:        print("⚠️ ConceptNet file not found. Download manually.")if os.path.exists(CN_PATH):    kg = ConceptNet(CN_PATH)    print(f"KG loaded: {len(kg.ent2id)} entities, {len(kg.rel2id)} relations")else:    kg = None    print("⚠️ KG not available, LLM-KGKAN will run without KG")

## Phase 0: Data Validation

In [ ]:
# Table 2 & 5: Dataset Statisticsstats = []for domain_key in DOMAIN_FILES:    s = get_dataset_stats(domain_key)    stats.append(s)    stats_df = pd.DataFrame(stats)print("\n📊 Dataset Statistics (Tables 2 & 5):")print(stats_df.to_string(index=False))print(f"\nTotal samples across all domains: {stats_df['total'].sum()}")

## Phase 1: Train BERT-Based Baselines\n\nBERT-UDA, AHF, TransProto, BGCA, KETGM, DALM

In [ ]:
bert_models = ["bert_uda", "ahf", "transproto", "bgca", "ketgm", "dalm"]# Standard benchmark (Table 1)for model_name in bert_models:    for src, tgt in STANDARD_PAIRS:        try:            print(f"\n{'='*50}")            print(f"{model_name}: {src} → {tgt}")            train_model(model_name, src, tgt, setting="standard")        except Exception as e:            print(f"[ERROR] {model_name} {src}→{tgt}: {e}")            continue    gc.collect()    torch.cuda.empty_cache()    print("\n✅ BERT baselines complete")

## Phase 1: Train Adapted Models (KGAN, SenticGCN)

In [ ]:
adapted_models = ["kgan", "senticgcn"]for model_name in adapted_models:    for src, tgt in STANDARD_PAIRS:        try:            print(f"\n{'='*50}")            print(f"{model_name}: {src} → {tgt}")            train_model(model_name, src, tgt, setting="standard")        except Exception as e:            print(f"[ERROR] {model_name} {src}→{tgt}: {e}")            continue    gc.collect()    torch.cuda.empty_cache()print("\n✅ Adapted models complete")

## Phase 1: Train LLMSynABSA (LLM-Augment-Syntax-DA)

In [ ]:
for src, tgt in STANDARD_PAIRS:    try:        print(f"\n{'='*50}")        print(f"llmsynabsa: {src} → {tgt}")        train_model("llmsynabsa", src, tgt, setting="standard")    except Exception as e:        print(f"[ERROR] llmsynabsa {src}→{tgt}: {e}")        continuegc.collect()torch.cuda.empty_cache()print("\n✅ LLMSynABSA complete")

## Phase 1: Train LLM-KGKAN (Main Model)

In [ ]:
for src, tgt in STANDARD_PAIRS:    try:        print(f"\n{'='*50}")        print(f"llm_kgkan: {src} → {tgt}")        train_model("llm_kgkan", src, tgt, setting="standard", kg=kg)    except Exception as e:        print(f"[ERROR] llm_kgkan {src}→{tgt}: {e}")        continuegc.collect()torch.cuda.empty_cache()print("\n✅ LLM-KGKAN complete")

## Phase 1: Few-Shot Training (Tables 3, 6)

In [ ]:
all_trainable = ["bert_uda", "ahf", "transproto", "bgca", "ketgm",                  "dalm", "kgan", "senticgcn", "llmsynabsa", "llm_kgkan"]for model_name in all_trainable:    for src, tgt in FEWSHOT_PAIRS:        try:            kwargs = {"kg": kg} if model_name == "llm_kgkan" else {}            train_model(model_name, src, tgt, setting="fewshot",                       k_shot=DEFAULT_FEWSHOT_K, **kwargs)        except Exception as e:            print(f"[ERROR] {model_name} {src}→{tgt} fewshot: {e}")    gc.collect(); torch.cuda.empty_cache()print("\n✅ Few-shot training complete")

## Phase 1: Zero-Shot Evaluation (Tables 3, 7)

In [ ]:
for model_name in all_trainable:    for src, tgt in ZEROSHOT_PAIRS:        try:            kwargs = {"kg": kg} if model_name == "llm_kgkan" else {}            train_model(model_name, src, tgt, setting="zeroshot",                       k_shot=0, **kwargs)        except Exception as e:            print(f"[ERROR] {model_name} {src}→{tgt} zeroshot: {e}")    gc.collect(); torch.cuda.empty_cache()print("\n✅ Zero-shot evaluation complete")

## Phase 2: LLM API Inference

In [ ]:
api_model_keys = list(API_MODELS.keys())for model_key in api_model_keys:    available, _ = check_api_key(model_key)    if not available:        print(f"[SKIP] {model_key}: No API key found — will show as blank in tables")        continue        print(f"\n{'='*50}")    print(f"API Inference: {model_key}")        # Standard pairs    for src, tgt in STANDARD_PAIRS:        existing = load_result(model_key, "standard", src, tgt)        if existing:            print(f"  [CACHED] {src}→{tgt}: {existing['macro_f1']}")            continue                tgt_data = UnifiedABSADataset(tgt, max_len=128)        samples = tgt_data.samples[:200]  # Limit for budget                tag_ids_list = run_api_inference(model_key, samples)                # Compute F1        preds_all, labels_all = [], []        for s, pred_ids in zip(samples, tag_ids_list):            if pred_ids is None:                continue            preds_all.extend(pred_ids)            labels_all.extend(s["tag_ids"][:len(pred_ids)])                if preds_all:            preds_t = torch.tensor(preds_all).unsqueeze(0)            labels_t = torch.tensor(labels_all).unsqueeze(0)            f1 = compute_macro_f1(preds_t, labels_t)            save_result(model_key, "standard", src, tgt, f1)            print(f"  {src}→{tgt}: F1={f1:.2f}")                if get_total_spent() >= API_BUDGET.max_budget_usd:            print(f"\n⚠️ Budget limit ${API_BUDGET.max_budget_usd} reached!")            break        if get_total_spent() >= API_BUDGET.max_budget_usd:        breakprint(f"\nTotal API spend: ${get_total_spent():.2f}")

## Phase 2: Few-Shot Sensitivity (Table 9)

In [ ]:
sensitivity_models = ["llm_kgkan", "llmsynabsa", "ketgm", "bert_uda"]for k in FEWSHOT_K_VALUES:    print(f"\n--- k = {k} ---")    for model_name in sensitivity_models:        for src, tgt in FEWSHOT_PAIRS[:4]:  # subset for speed            try:                kwargs = {"kg": kg} if model_name == "llm_kgkan" else {}                train_model(model_name, src, tgt,                            setting=f"fewshot_k{k}", k_shot=k, **kwargs)            except Exception as e:                print(f"[ERROR] {model_name} {src}→{tgt} k={k}: {e}")        gc.collect(); torch.cuda.empty_cache()print("\n✅ Sensitivity analysis complete")

## Phase 2: LLM-KGKAN Ablation Studies (Tables 12, 13)

In [ ]:
# Component ablationablation_pairs = STANDARD_PAIRS[:4]  # subset for speedfor variant in ["wo_kg", "wo_syn", "wo_arg", "wo_kan"]:    print(f"\n--- Ablation: {variant} ---")    for src, tgt in ablation_pairs:        try:            train_model("llm_kgkan", src, tgt, setting="ablation",                       kg=kg, variant=variant)        except Exception as e:            print(f"[ERROR] ablation {variant} {src}→{tgt}: {e}")    gc.collect(); torch.cuda.empty_cache()print("\n✅ Ablation studies complete")

---\n## Phase 3: Results — Tables

### Table 1: Standard Cross-Domain ABSA Benchmark

In [ ]:
t1_models = TABLE1_MODELSt1_df = build_table_df(t1_models, STANDARD_PAIRS, "standard")print("Table 1: Standard Cross-Domain ABSA (Macro-F1 %)")print(t1_df.to_string(index=False))t1_df.to_csv("results/table1.csv", index=False)

### Table 2: Dataset Statistics

In [ ]:
stats = [get_dataset_stats(d) for d in ["L", "R", "D", "S"]]t2_df = pd.DataFrame(stats)print("Table 2: Dataset Statistics (Source Domains)")print(t2_df.to_string(index=False))

### Table 3: Low-Resource & No-Label Targets

In [ ]:
t3_fs = build_table_df(TABLE1_MODELS, FEWSHOT_PAIRS, "fewshot")t3_zs = build_table_df(TABLE1_MODELS, ZEROSHOT_PAIRS, "zeroshot")print("Table 3a: Few-Shot (k=16)")print(t3_fs.to_string(index=False))print("\nTable 3b: Zero-Shot")print(t3_zs.to_string(index=False))t3_fs.to_csv("results/table3_fewshot.csv", index=False)t3_zs.to_csv("results/table3_zeroshot.csv", index=False)

### Table 4: Hyperparameters

In [ ]:
print("Table 4: Hyperparameters")cfg = LLMKGKANModelConfig()hyper = {    "Backbone": cfg.llm_name,    "LoRA r": PEFT.lora_r, "LoRA α": PEFT.lora_alpha,    "Batch Size": GPU.batch_size_llm, "LR": cfg.lr,    "Epochs": cfg.epochs, "Max Len": cfg.max_len,    "KG Emb Dim": cfg.kg_emb_dim, "RGCN Layers": cfg.rgcn_layers,    "MMD λ": cfg.mmd_lambda, "Dropout": cfg.dropout,    "Seed": SEED,}for k, v in hyper.items():    print(f"  {k}: {v}")

### Table 5: Extended Dataset Statistics

In [ ]:
stats5 = [get_dataset_stats(d) for d in DOMAIN_FILES.keys()]t5_df = pd.DataFrame(stats5)print("Table 5: All Domain Statistics")print(t5_df.to_string(index=False))t5_df.to_csv("results/table5.csv", index=False)

### Table 6: Detailed Few-Shot Pairwise

In [ ]:
t6_df = build_table_df(TABLE1_MODELS, FEWSHOT_PAIRS, "fewshot")print("Table 6: Few-Shot Pairwise Results (k=16)")print(t6_df.to_string(index=False))t6_df.to_csv("results/table6.csv", index=False)

### Table 7: Detailed Zero-Shot Pairwise

In [ ]:
t7_df = build_table_df(TABLE1_MODELS, ZEROSHOT_PAIRS, "zeroshot")print("Table 7: Zero-Shot Pairwise Results")print(t7_df.to_string(index=False))t7_df.to_csv("results/table7.csv", index=False)

### Table 8: Statistical Significance (p-values)

In [ ]:
from scipy import stats as sp_statsbaselines = ["bert_uda", "ahf", "transproto", "bgca", "ketgm", "dalm", "llmsynabsa"]proposed = "llm_kgkan"print("Table 8: Paired t-test (LLM-KGKAN vs baselines)")print(f"{'Baseline':<25} {'t-stat':>8} {'p-value':>10} {'Significant':>12}")print("-" * 60)for bl in baselines:    bl_vals, pr_vals = [], []    for src, tgt in STANDARD_PAIRS:        rb = load_result(bl, "standard", src, tgt)        rp = load_result(proposed, "standard", src, tgt)        if rb and rp:            bl_vals.append(rb["macro_f1"])            pr_vals.append(rp["macro_f1"])    if len(bl_vals) >= 2:        t_stat, p_val = sp_stats.ttest_rel(pr_vals, bl_vals)        sig = "Yes" if p_val < 0.05 else "No"        bl_name = MODEL_DISPLAY_NAMES.get(bl, bl)        print(f"{bl_name:<25} {t_stat:>8.3f} {p_val:>10.4f} {sig:>12}")    else:        print(f"{bl:<25} {'N/A':>8} {'N/A':>10} {'N/A':>12}")

### Table 9: Few-Shot Sensitivity

In [ ]:
print("Table 9: Few-Shot Sensitivity (Macro-F1 %)")rows = []for model_name in ["llm_kgkan", "llmsynabsa", "ketgm", "bert_uda"]:    row = {"Model": MODEL_DISPLAY_NAMES.get(model_name, model_name)}    for k in FEWSHOT_K_VALUES:        vals = []        for src, tgt in FEWSHOT_PAIRS[:4]:            r = load_result(model_name, f"fewshot_k{k}", src, tgt)            if r:                vals.append(r["macro_f1"])        row[f"k={k}"] = round(np.mean(vals), 2) if vals else None    rows.append(row)t9_df = pd.DataFrame(rows)print(t9_df.to_string(index=False))t9_df.to_csv("results/table9.csv", index=False)

### Table 10: Target-Wise Few-Shot (LLM-KGKAN)

In [ ]:
print("Table 10: LLM-KGKAN Few-Shot by Target Domain")rows = []for tgt in LOW_RESOURCE_TARGETS:    row = {"Target": tgt}    for src in SOURCE_DOMAINS:        r = load_result("llm_kgkan", "fewshot", src, tgt)        row[f"from {src}"] = r["macro_f1"] if r else None    vals = [v for k, v in row.items() if k != "Target" and v is not None]    row["AVG"] = round(np.mean(vals), 2) if vals else None    rows.append(row)t10_df = pd.DataFrame(rows)print(t10_df.to_string(index=False))

### Table 11: Efficiency Comparison

In [ ]:
print("Table 11: Efficiency Comparison")eff_data = [    {"Model": "BERT-UDA", "Params (M)": 110, "Train Time (min)": "~5", "Backbone": "BERT-base"},    {"Model": "AHF", "Params (M)": 112, "Train Time (min)": "~8", "Backbone": "BERT-base"},    {"Model": "TransProto", "Params (M)": 110, "Train Time (min)": "~10", "Backbone": "BERT-base"},    {"Model": "BGCA", "Params (M)": 139, "Train Time (min)": "~15", "Backbone": "BART-base"},    {"Model": "KETGM", "Params (M)": 115, "Train Time (min)": "~20", "Backbone": "BERT-base"},    {"Model": "DALM", "Params (M)": 124, "Train Time (min)": "~12", "Backbone": "GPT-2"},    {"Model": "LLM-Augment-Syn-DA", "Params (M)": "~8000", "Train Time (min)": "~45", "Backbone": "LLaMA-3-8B"},    {"Model": "KGAN", "Params (M)": 112, "Train Time (min)": "~15", "Backbone": "BERT-base"},    {"Model": "SenticGCN", "Params (M)": 113, "Train Time (min)": "~10", "Backbone": "BERT-base"},    {"Model": "LLM-KGKAN", "Params (M)": "~8000", "Train Time (min)": "~60", "Backbone": "LLaMA-3-8B"},]t11_df = pd.DataFrame(eff_data)print(t11_df.to_string(index=False))

### Tables 12-16

In [ ]:
# Table 12: Fusion ablationprint("Table 12: Fusion Strategy Ablation")fusion_rows = []for variant in ["concat_mlp", "weighted_sum", "gated_fusion", "bilinear_fusion",                 "cross_attention", "kan_fusion"]:    vals = []    for src, tgt in STANDARD_PAIRS[:4]:        r = load_result(f"llm_kgkan_{variant}", "ablation", src, tgt)        if r: vals.append(r["macro_f1"])    fusion_rows.append({"Fusion": variant, "AVG F1": round(np.mean(vals), 2) if vals else "—"})print(pd.DataFrame(fusion_rows).to_string(index=False))# Table 13: KG source ablationprint("\nTable 13: KG Source Ablation")kg_rows = []for kg_src in ["none", "wordnet", "senticnet", "conceptnet", "hybrid"]:    vals = []    for src, tgt in STANDARD_PAIRS[:4]:        r = load_result(f"llm_kgkan_{kg_src}", "ablation", src, tgt)        if r: vals.append(r["macro_f1"])    kg_rows.append({"KG Source": kg_src, "AVG F1": round(np.mean(vals), 2) if vals else "—"})print(pd.DataFrame(kg_rows).to_string(index=False))# Table 14-16: Static tables from paperprint("\nTable 14: Qualitative Error Analysis — see paper Section 5.5")print("Table 15: Remaining Failure Cases — see paper Section 5.5")print("Table 16: KG Relation Type Distribution — see paper Section 5.6")

---\n## Phase 3: Results — Figures

In [ ]:
# Figure 3/4: Few-shot sensitivityfig1 = fig_fewshot_sensitivity(    ["llm_kgkan", "llmsynabsa", "ketgm", "bert_uda"],    LOW_RESOURCE_TARGETS,    save_path="results/fig_fewshot_sensitivity.png")plt.show()print("Saved: results/fig_fewshot_sensitivity.png")

In [ ]:
# Figure 5: Error distributionfig2 = fig_error_distribution(    ["llm_kgkan", "llmsynabsa", "ketgm", "bert_uda"],    STANDARD_PAIRS, "standard",    save_path="results/fig_error_distribution.png")plt.show()print("Saved: results/fig_error_distribution.png")

In [ ]:
# Figure 6: Model gainfig3 = fig_model_gain(    "llmsynabsa", "llm_kgkan", STANDARD_PAIRS, "standard",    save_path="results/fig_model_gain.png")plt.show()print("Saved: results/fig_model_gain.png")

In [ ]:
# Figure 7: Transfer heatmapfig4 = fig_transfer_heatmap(    "llm_kgkan", STANDARD_PAIRS + FEWSHOT_PAIRS + ZEROSHOT_PAIRS,    "standard",    save_path="results/fig_transfer_heatmap.png")plt.show()print("Saved: results/fig_transfer_heatmap.png")

---\n## Summary

In [ ]:
print("=" * 60)print("REPRODUCTION COMPLETE")print("=" * 60)print(f"\nResults saved to: {RESULTS_DIR}/")print(f"Models saved to: {SAVED_MODELS_DIR}/")print(f"API spend: ${get_total_spent():.2f} / ${API_BUDGET.max_budget_usd}")print(f"\nGenerated files:")for f in sorted(os.listdir("results")):    if f.endswith((".csv", ".png")):        print(f"  results/{f}")